# 01 - Local Detection and Tracking
يشغّل YOLOv8 + ByteTrack لكل فيديو كاميرا ويكتب detections مجهولة الهوية في `Output/tables/local_tracks.csv`.

### Interpolation
يدعم Foot-Point Interpolation وقت الرسم فقط:
لو track اختفى لأقل من `INTERP_MAX_GAP` فريم ورجع تاني بنفس الـ ID، النظام بيحسب
موضع تقديري للبوكس في الفريمات الناقصة بخط مستقيم (Linear Interpolation).
ده **لا يؤثر** على ملف `local_tracks.csv` — البيانات الحقيقية تفضل كما هي.

In [ ]:
from __future__ import annotations

from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
import json

import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

# ──────────────────────────────────── paths ────────────────────────────────────
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebook':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'Data' / 'raw'
TABLES_DIR = PROJECT_ROOT / 'Output' / 'tables'
VIDEOS_DIR = PROJECT_ROOT / 'Output' / 'videos'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

# ──────────────────────────────────── config ───────────────────────────────────
#yolo11l.pt is more stronger model
MODEL_NAME = 'yolo11m.pt'  # change only after the baseline works
MODEL_PATH = PROJECT_ROOT / MODEL_NAME
if not MODEL_PATH.exists():
    MODEL_PATH = PROJECT_ROOT / 'Notebook' / MODEL_NAME
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'YOLO weights not found locally: {MODEL_NAME}')
# Feed low-confidence detections to ByteTrack so an existing identity can survive
# a short occlusion. New tracks still require 0.25 in the retail tracker config.
CONFIDENCE = 0.40
MIN_SAVED_CONFIDENCE = 0.25
TRACKER_PATH = PROJECT_ROOT / 'Data' / 'config' / 'bytetrack_retail.yaml'
if not TRACKER_PATH.exists():
    raise FileNotFoundError(f'Missing tracker config: {TRACKER_PATH}')
FRAME_STRIDE = 1  # increase to 2 or 3 for faster experiments
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv'}
POINT_COLOR = (255, 0, 0)  # OpenCV BGR: fixed blue
INTERP_COLOR = (80, 180, 255)  # light blue for interpolated foot points
TEXT_COLOR = (255, 255, 255)

# ──── Interpolation config ────
# Maximum gap (in frames) to fill with interpolated foot points.
# If a track disappears for more than this many frames, no interpolation.
INTERP_MAX_GAP = 10


# ──────────────────────────── helpers ──────────────────────────────────────────

def select_inference_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        from torchvision.ops import nms
        boxes = torch.tensor([[0.0, 0.0, 10.0, 10.0]], device='cuda')
        scores = torch.tensor([0.9], device='cuda')
        nms(boxes, scores, 0.5)
        return 0
    except (RuntimeError, NotImplementedError, ImportError, AttributeError) as error:
        print(f'CUDA NMS is unavailable ({type(error).__name__}); using CPU safely.')
        return 'cpu'


def create_video_writer(output_path, frame, fps):
    height, width = frame.shape[:2]
    codec = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(output_path), codec, fps, (width, height))
    if not writer.isOpened():
        writer.release()
        raise RuntimeError(f'Could not create annotated video: {output_path}')
    return writer


def read_video_metadata(video_path):
    """Read the real frame-based duration used by later analytics."""
    capture = cv2.VideoCapture(str(video_path))
    fps = capture.get(cv2.CAP_PROP_FPS)
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    capture.release()
    if not fps or fps <= 0:
        fps = 25.0
    if total_frames <= 0:
        raise RuntimeError(f'Could not read frame count from {video_path.name}')
    return {
        'fps': float(fps),
        'total_frames': total_frames,
        'duration_sec': total_frames / fps,
    }


def draw_tracking_annotations(frame, foot_points, track_ids, point_color=POINT_COLOR):
    """Draw one labeled foot-point per tracked person."""
    annotated_frame = frame.copy()
    frame_height, frame_width = annotated_frame.shape[:2]
    for point, local_track_id in zip(foot_points, track_ids):
        foot_x, foot_y = (int(round(value)) for value in point)
        foot_x = max(0, min(foot_x, frame_width - 1))
        foot_y = max(0, min(foot_y, frame_height - 1))
        cv2.circle(annotated_frame, (foot_x, foot_y), 10, point_color, -1, cv2.LINE_AA)
        cv2.circle(annotated_frame, (foot_x, foot_y), 12, TEXT_COLOR, 1, cv2.LINE_AA)
        label = f'LID {int(local_track_id)}'
        label_x = min(foot_x + 9, max(0, frame_width - 70))
        label_y = max(foot_y - 8, 20)
        cv2.putText(
            annotated_frame,
            label,
            (label_x, label_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            TEXT_COLOR,
            2,
            cv2.LINE_AA,
        )
    return annotated_frame


# ───────────────────────── Foot-Point Interpolation ────────────────────────────


class TrackInterpolator:
    """Buffer-based foot-point interpolator for annotation only.

    The idea:
    - We buffer up to `max_gap + 1` frames before writing them to the video.
    - When a track reappears after a gap ≤ max_gap, we go back through the
      buffer and draw linearly-interpolated foot points on the frames where the
      track was missing.
    - This has **zero effect** on local_tracks.csv — only the annotated video
      gets the extra interpolated points.
    """

    def __init__(self, max_gap: int = INTERP_MAX_GAP):
        self.max_gap = max_gap
        # track_id -> (last_seen_frame_index, [foot_x, foot_y])
        self.last_seen: dict[int, tuple[int, np.ndarray]] = {}
        # Ring-buffer of (frame_index, annotated_frame) tuples
        self.frame_buffer: list[tuple[int, np.ndarray]] = []

    def _buffer_index(self, target_frame_index: int) -> int | None:
        """Return the buffer position for a given frame_index, or None."""
        for i, (fi, _) in enumerate(self.frame_buffer):
            if fi == target_frame_index:
                return i
        return None

    def feed(self, frame_index: int, annotated_frame: np.ndarray,
             foot_points: list | np.ndarray, track_ids: list[int]) -> list[np.ndarray]:
        """Accept a new frame and its detections.

        Returns a (possibly empty) list of annotated frames that are ready to
        be written to the video writer.  The caller should write them in order.
        """
        # ── Step 1: Interpolate any tracks that reappeared after a gap ──
        for point, tid in zip(foot_points, track_ids):
            point = np.asarray(point, dtype=np.float64)
            if tid in self.last_seen:
                prev_fi, prev_point = self.last_seen[tid]
                gap = frame_index - prev_fi  # number of frames since last seen
                if 1 < gap <= self.max_gap + 1:
                    # We have a fillable gap — interpolate the missing foot points
                    for step in range(1, gap):
                        t = step / gap  # 0..1 parameter
                        interp_point = prev_point * (1 - t) + point * t
                        missing_fi = prev_fi + step
                        buf_idx = self._buffer_index(missing_fi)
                        if buf_idx is not None:
                            # Draw the interpolated foot point onto the buffered frame
                            self.frame_buffer[buf_idx] = (
                                missing_fi,
                                draw_tracking_annotations(
                                    self.frame_buffer[buf_idx][1],
                                    [interp_point],
                                    [tid],
                                    point_color=INTERP_COLOR,
                                ),
                            )
            # Update last-seen
            self.last_seen[tid] = (frame_index, point)

        # ── Step 2: Add current frame to the buffer ──
        self.frame_buffer.append((frame_index, annotated_frame))

        # ── Step 3: Flush frames that are old enough ──
        flush_cutoff = frame_index - self.max_gap
        ready: list[np.ndarray] = []
        while self.frame_buffer and self.frame_buffer[0][0] <= flush_cutoff:
            _, flushed_frame = self.frame_buffer.pop(0)
            ready.append(flushed_frame)
        return ready

    def flush_remaining(self) -> list[np.ndarray]:
        """Flush everything left in the buffer (call at end of video)."""
        remaining = [frame for _, frame in self.frame_buffer]
        self.frame_buffer.clear()
        return remaining


# ──────────────────────────── main loop ─────────────────────────────────────

DEVICE = select_inference_device()
print(f'Inference device: {DEVICE}')
# Only process videos placed directly in Data/raw; ignore subfolders such as more_came.
videos = sorted(path for path in RAW_DIR.iterdir() if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS)
if not videos:
    raise FileNotFoundError('Add camera videos to Data/raw, e.g. camera_01.mp4')

employee_ids = json.loads((PROJECT_ROOT / 'Data' / 'config' / 'employee_local_ids.json').read_text(encoding='utf-8'))
model = YOLO(str(MODEL_PATH))
rows = []
video_metadata_rows = []

for video_path in videos:
    camera_id = video_path.stem
    video_metadata = read_video_metadata(video_path)
    fps = video_metadata['fps']
    video_metadata_rows.append({
        'camera_id': camera_id,
        'source_video': video_path.relative_to(RAW_DIR).as_posix(),
        **video_metadata,
    })
    staff_ids = set(employee_ids.get(camera_id, []))
    annotated_path = VIDEOS_DIR / f'annotated_{camera_id}.mp4'
    print(f'Processing {video_path.name} at {fps:.2f} FPS')

    results = model.track(
        source=str(video_path),
        stream=True,
        # False resets tracker state when the source video changes.  Keeping
        # this True leaked Cam17 IDs into Cam18, etc.
        persist=False,
        classes=[0],  # person only
        conf=CONFIDENCE,
        tracker=str(TRACKER_PATH),
        device=DEVICE,
        verbose=False,
    )
    writer = None
    interpolator = TrackInterpolator(max_gap=INTERP_MAX_GAP)

    try:
        for frame_index, result in enumerate(results):
            frame = result.orig_img
            if frame is None:
                continue

            boxes = result.boxes
            if boxes is None or boxes.id is None:
                xyxy = []
                track_ids = []
                confidences = []
            else:
                xyxy = boxes.xyxy.cpu().numpy()
                track_ids = boxes.id.int().cpu().tolist()
                confidences = boxes.conf.cpu().numpy()

            # Draw one foot point per real detection to keep crowded scenes readable
            foot_points = [
                ((float(x1) + float(x2)) / 2, (float(y1) + float(y2)) / 2)
                for x1, y1, x2, y2 in xyxy
            ]
            annotated_frame = draw_tracking_annotations(frame, foot_points, track_ids)

            if writer is None:
                writer = create_video_writer(annotated_path, annotated_frame, fps)

            # Feed to interpolator — it may return frames ready to write
            ready_frames = interpolator.feed(frame_index, annotated_frame, foot_points, track_ids)
            for rf in ready_frames:
                writer.write(rf)

            # ── CSV rows (unchanged — real detections only) ──
            if frame_index % FRAME_STRIDE or not track_ids:
                continue
            for box, local_track_id, confidence in zip(xyxy, track_ids, confidences):
                if float(confidence) < MIN_SAVED_CONFIDENCE:
                    continue
                x1, y1, x2, y2 = map(float, box)
                rows.append({
                    'camera_id': camera_id,
                    'frame_index': frame_index,
                    'timestamp_sec': round(frame_index / fps, 3),
                    'local_track_id': int(local_track_id),
                    'is_employee': int(local_track_id) in staff_ids,
                    'confidence': round(float(confidence), 4),
                    'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                    'foot_x': round((x1 + x2) / 2, 2),
                    'foot_y': round(y2, 2),
                })

        # Flush any remaining buffered frames
        for rf in interpolator.flush_remaining():
            if writer is not None:
                writer.write(rf)

    finally:
        if writer is not None:
            writer.release()

    if writer is None:
        raise RuntimeError(f'No frames could be read from {video_path.name}')
    print(f'Saved annotated video to {annotated_path.relative_to(PROJECT_ROOT)}')

local_tracks = pd.DataFrame(rows)
if local_tracks.empty:
    raise RuntimeError('No people were detected. Check the video, model weights, and CONFIDENCE.')
output_path = TABLES_DIR / 'local_tracks.csv'
local_tracks.to_csv(output_path, index=False)
metadata_path = TABLES_DIR / 'video_metadata.csv'
pd.DataFrame(video_metadata_rows).to_csv(metadata_path, index=False)
manifest_path = TABLES_DIR / 'local_tracking_run.json'
manifest_path.write_text(json.dumps({
    'schema_version': 1,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_name': MODEL_NAME,
    'confidence': CONFIDENCE,
    'minimum_saved_confidence': MIN_SAVED_CONFIDENCE,
    'tracker': TRACKER_PATH.relative_to(PROJECT_ROOT).as_posix(),
    'tracker_persist': False,
    'id_scope': 'camera_local',
    'camera_ids': [row['camera_id'] for row in video_metadata_rows],
}, indent=2), encoding='utf-8')
print(f'Saved {len(local_tracks):,} detections to {output_path.relative_to(PROJECT_ROOT)}')
print(f'Saved frame-based video metadata to {metadata_path.relative_to(PROJECT_ROOT)}')
print(f'Saved per-camera tracker manifest to {manifest_path.relative_to(PROJECT_ROOT)}')
local_tracks.head()

Inference device: 0
Processing CAFE_place_05_camera_17_15min.mp4 at 5.00 FPS
Saved annotated video to Output\videos\annotated_CAFE_place_05_camera_17_15min.mp4
Processing CAFE_place_05_camera_18_15min.mp4 at 5.00 FPS
Saved annotated video to Output\videos\annotated_CAFE_place_05_camera_18_15min.mp4
Saved 93,362 detections to Output\tables\local_tracks.csv
Saved frame-based video metadata to Output\tables\video_metadata.csv
Saved per-camera tracker manifest to Output\tables\local_tracking_run.json


,camera_id,frame_index,timestamp_sec,local_track_id,is_employee,confidence,x1,y1,x2,y2,foot_x,foot_y
0,CAFE_place_05_camera_17_15min,0,0.0,1,False,0.8952,434.930664,485.800171,728.544678,963.834473,581.74,963.83
1,CAFE_place_05_camera_17_15min,0,0.0,2,False,0.8829,1257.209229,426.466187,1491.431519,696.190796,1374.32,696.19
2,CAFE_place_05_camera_17_15min,0,0.0,3,False,0.8336,1733.588013,400.347900,1903.747925,848.694092,1818.67,848.69
3,CAFE_place_05_camera_17_15min,0,0.0,4,False,0.8104,701.803833,304.082764,821.142334,525.334595,761.47,525.33
4,CAFE_place_05_camera_17_15min,0,0.0,5,False,0.7756,1241.905029,363.487305,1342.502808,548.302979,1292.20,548.30


**Privacy note:** this notebook stores bounding boxes and anonymous tracker IDs only. Do not add face images or identity fields to the exported table.